# 1. Whole-Leaf Feature Extraction

**Project:** Orange Leaf Nutrient Deficiency and Disease Classification Using Machine Learning

This version uses the **whole-leaf region file** instead of the circular ROI.

The region file identifies the complete leaf area, so the extracted features use information from the whole leaf rather than only an internal circular area.

### Output

The extracted features are saved as:

`data/citrus_features_8_classes.csv`

EDA is intentionally kept in a separate notebook.

## 1.1 Import Libraries

In [9]:
from pathlib import Path

import cv2
import numpy as np
import pandas as pd

print("Libraries imported successfully.")

Libraries imported successfully.


## 1.2 Dataset Paths

```text
project_folder/
├── CitrusUAT_Selected_Dataset/
│   ├── Images/
│   └── Masks/
└── data/
```

The image and region filenames should match, for example:

```text
Fe_001.jpg
Fe_001_mask.png
```

In [10]:
DATASET_ROOT = Path("CitrusUAT_Selected_Dataset")

IMAGE_DIR = DATASET_ROOT / "Images"
REGION_DIR = DATASET_ROOT / "Masks"

DATA_DIR = Path("data")
DATA_DIR.mkdir(exist_ok=True)

OUTPUT_CSV = DATA_DIR / "citrus_features_8_classes.csv"

SELECTED_CLASSES = [
    "Healthy",
    "Fe",
    "Mg",
    "Mn",
    "N",
    "Zn",
    "HLB",
    "Greasy_spot"
]

print("Images:", IMAGE_DIR.resolve())
print("Leaf-region files:", REGION_DIR.resolve())
print("Output:", OUTPUT_CSV.resolve())

Images: C:\Users\Shakir Ali\Downloads\Orange_Leaf_ML_Project\CitrusUAT_Selected_Dataset\Images
Leaf-region files: C:\Users\Shakir Ali\Downloads\Orange_Leaf_ML_Project\CitrusUAT_Selected_Dataset\Masks
Output: C:\Users\Shakir Ali\Downloads\Orange_Leaf_ML_Project\data\citrus_features_8_classes.csv


## 1.3 File Helper Functions

The filename is used only to identify the class and locate the matching leaf-region file.

It is **not** used as a model feature.

In [11]:
def get_class_from_filename(stem):
    """Get class label from an image filename."""

    ordered_classes = [
        "Greasy_spot",
        "Healthy",
        "HLB",
        "Fe",
        "Mg",
        "Mn",
        "N",
        "Zn"
    ]

    for class_name in ordered_classes:
        if stem == class_name or stem.startswith(class_name + "_"):
            return class_name

    return None


def find_corresponding_region(image_path):
    """Find the whole-leaf region file that belongs to an image."""

    stem = image_path.stem

    for extension in [".png", ".jpg", ".jpeg", ".PNG", ".JPG", ".JPEG"]:

        candidate = REGION_DIR / f"{stem}_mask{extension}"

        if candidate.exists():
            return candidate

    return None

## 1.4 Feature Extraction

The same 17 simple features are extracted as before.

### Overall color
- mean RGB
- mean HSV

### Color variation
- standard deviation of RGB
- standard deviation of HSV
- grayscale standard deviation

### Color proportions
- green ratio
- yellow ratio
- dark ratio

### Simple structure
- edge density

All measurements now come from the complete leaf area.

In [12]:
def extract_features(image_path, region_path):
    """Extract 17 features from the complete leaf region."""

    # ---------- Load image ----------
    image_bgr = cv2.imread(str(image_path))

    if image_bgr is None:
        raise ValueError(f"Could not read image: {image_path}")

    image_rgb = cv2.cvtColor(
        image_bgr,
        cv2.COLOR_BGR2RGB
    )

    # ---------- Load whole-leaf region file ----------
    region_image = cv2.imread(
        str(region_path),
        cv2.IMREAD_GRAYSCALE
    )

    if region_image is None:
        raise ValueError(f"Could not read region file: {region_path}")

    if region_image.shape != image_rgb.shape[:2]:
        region_image = cv2.resize(
            region_image,
            (image_rgb.shape[1], image_rgb.shape[0]),
            interpolation=cv2.INTER_NEAREST
        )

    leaf_region = region_image > 127

    if not np.any(leaf_region):
        raise ValueError(f"Region file contains no leaf pixels: {region_path}")

    # ---------- RGB ----------
    rgb_pixels = image_rgb[leaf_region].astype(np.float32)

    R = rgb_pixels[:, 0]
    G = rgb_pixels[:, 1]
    B = rgb_pixels[:, 2]

    # ---------- HSV ----------
    # H = hue: basic color type
    # S = saturation: color strength
    # V = value: brightness
    hsv_image = cv2.cvtColor(
        image_rgb,
        cv2.COLOR_RGB2HSV
    )

    hsv_pixels = hsv_image[leaf_region].astype(np.float32)

    H = hsv_pixels[:, 0]
    S = hsv_pixels[:, 1]
    V = hsv_pixels[:, 2]

    # ---------- Color ratios ----------
    green_pixels = (
        (H >= 35) &
        (H <= 90) &
        (S >= 40)
    )

    yellow_pixels = (
        (H >= 20) &
        (H < 35) &
        (S >= 40)
    )

    dark_pixels = V < 100

    # ---------- Grayscale variation ----------
    gray_image = cv2.cvtColor(
        image_rgb,
        cv2.COLOR_RGB2GRAY
    )

    gray_pixels = gray_image[leaf_region].astype(np.float32)

    # ---------- Edge density ----------
    edges = cv2.Canny(
        gray_image,
        100,
        200
    )

    features = {
        "mean_R": np.mean(R),
        "mean_G": np.mean(G),
        "mean_B": np.mean(B),

        "std_R": np.std(R),
        "std_G": np.std(G),
        "std_B": np.std(B),

        "mean_H": np.mean(H),
        "mean_S": np.mean(S),
        "mean_V": np.mean(V),

        "std_H": np.std(H),
        "std_S": np.std(S),
        "std_V": np.std(V),

        "green_ratio": np.mean(green_pixels),
        "yellow_ratio": np.mean(yellow_pixels),
        "dark_ratio": np.mean(dark_pixels),

        "gray_std": np.std(gray_pixels),

        "edge_density": np.mean(
            edges[leaf_region] > 0
        )
    }

    return features

## 1.5 Process All Selected Images

In [13]:
rows = []

image_paths = []

for pattern in [
    "*.jpg", "*.jpeg", "*.png",
    "*.JPG", "*.JPEG", "*.PNG"
]:
    image_paths.extend(IMAGE_DIR.glob(pattern))

image_paths = sorted(set(image_paths))

processed = 0
skipped = 0

for image_path in image_paths:

    class_name = get_class_from_filename(
        image_path.stem
    )

    if class_name not in SELECTED_CLASSES:
        continue

    region_path = find_corresponding_region(
        image_path
    )

    if region_path is None:
        print("Missing region file:", image_path.name)
        skipped += 1
        continue

    try:
        features = extract_features(
            image_path,
            region_path
        )

        features["image_name"] = image_path.name
        features["class"] = class_name

        rows.append(features)
        processed += 1

    except Exception as error:
        print(
            f"Skipped {image_path.name}: {error}"
        )
        skipped += 1

print("Processed:", processed)
print("Skipped:", skipped)

Processed: 623
Skipped: 0


## 1.6 Validate and Save the Feature Dataset

This is only a data-quality check.

The EDA notebook handles the actual visual analysis.

In [14]:
FEATURE_COLUMNS = [
    "mean_R", "mean_G", "mean_B",
    "std_R", "std_G", "std_B",
    "mean_H", "mean_S", "mean_V",
    "std_H", "std_S", "std_V",
    "green_ratio", "yellow_ratio", "dark_ratio",
    "gray_std", "edge_density"
]

df = pd.DataFrame(rows)

df = df[
    ["image_name", "class"] + FEATURE_COLUMNS
]

print("Dataset shape:", df.shape)

print("\nClass counts:")
print(df["class"].value_counts())

print(
    "\nMissing values:",
    df[FEATURE_COLUMNS].isna().sum().sum()
)

print(
    "Infinite values:",
    np.isinf(
        df[FEATURE_COLUMNS].to_numpy()
    ).sum()
)

df.to_csv(
    OUTPUT_CSV,
    index=False
)

print("\nSaved:", OUTPUT_CSV.resolve())

Dataset shape: (623, 19)

Class counts:
class
Fe             100
Greasy_spot    100
Healthy        100
Mg             100
Zn             100
N               50
HLB             43
Mn              30
Name: count, dtype: int64

Missing values: 0
Infinite values: 0

Saved: C:\Users\Shakir Ali\Downloads\Orange_Leaf_ML_Project\data\citrus_features_8_classes.csv
